In [97]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [98]:
numere = np.array([1, 100, 125, 500, 1000]).reshape(-1, 1)
min_max_scaler = MinMaxScaler()
min_max_scaler



,feature_range,"(0, ...)"
,copy,True
,clip,False


In [99]:
min_max_scaler.fit_transform(numere)

array([[0.        ],
       [0.0990991 ],
       [0.12412412],
       [0.4994995 ],
       [1.        ]])

In [100]:
from sklearn.preprocessing import StandardScaler

In [101]:
standard_scaler = StandardScaler()
standard_scaler

,copy,True
,with_mean,True
,with_std,True


In [102]:
standard_scaler.fit_transform(numere)

array([[-0.93347317],
       [-0.66498437],
       [-0.59718417],
       [ 0.41981884],
       [ 1.77582286]])

# 1. Pentru text - One Hot Encoder

### Fiecare categorie devine o coloană binară

In [103]:
df = pd.DataFrame({
    "culoare":["rosu", "albastru", "verde", "rosu", "verde"],
    "marime" : ["S", "M", "L", "M", "S"]
})
df

,culoare,marime
0,rosu,S
1,albastru,M
2,verde,L
3,rosu,M
4,verde,S


In [104]:
pd.get_dummies(df["culoare"])

,albastru,rosu,verde
0,False,True,False
1,True,False,False
2,False,False,True
3,False,True,False
4,False,False,True


In [105]:
pd.get_dummies(df["culoare"], dtype=int)

,albastru,rosu,verde
0,0,1,0
1,1,0,0
2,0,0,1
3,0,1,0
4,0,0,1


In [106]:
pd.get_dummies(df["culoare"], prefix="culoare", dtype=int)

,culoare_albastru,culoare_rosu,culoare_verde
0,0,1,0
1,1,0,0
2,0,0,1
3,0,1,0
4,0,0,1


# One hot encoder -> Sklearn 

In [107]:
from sklearn.preprocessing import OneHotEncoder

In [108]:
ohe = OneHotEncoder()
ohe

,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [109]:
df

,culoare,marime
0,rosu,S
1,albastru,M
2,verde,L
3,rosu,M
4,verde,S


In [110]:
ohe.fit(df)

,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [111]:
# Set-urile din categorii
ohe.categories_

[array(['albastru', 'rosu', 'verde'], dtype=object),
 array(['L', 'M', 'S'], dtype=object)]

In [112]:
df_ohe = ohe.transform(df)
df_ohe

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10 stored elements and shape (5, 6)>

In [113]:
df_ohe.toarray()

array([[0., 1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1., 0.],
       [0., 0., 1., 1., 0., 0.],
       [0., 1., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 1.]])

In [114]:
coloane = ohe.get_feature_names_out()
coloane

array(['culoare_albastru', 'culoare_rosu', 'culoare_verde', 'marime_L',
       'marime_M', 'marime_S'], dtype=object)

In [115]:
coloane = ohe.get_feature_names_out()
pd.DataFrame(df_ohe.toarray(), columns=coloane)

,culoare_albastru,culoare_rosu,culoare_verde,marime_L,marime_M,marime_S
0,0.0,1.0,0.0,0.0,0.0,1.0
1,1.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,1.0,1.0,0.0,0.0
3,0.0,1.0,0.0,0.0,1.0,0.0
4,0.0,0.0,1.0,0.0,0.0,1.0


# 2. Bag of Words -> BoW

In [116]:
corpus = [
    "Pisica mananca peste",
    "Copilul mananca iaurt",
    "Copilul si pisica se joaca",
]
corpus

['Pisica mananca peste', 'Copilul mananca iaurt', 'Copilul si pisica se joaca']

### 2.1. Varianta Manuală

In [117]:
tokens =  [doc.lower().split() for doc in corpus]
words = [word for document in tokens for word in document ]
words = list(set(words))
words

['joaca', 'se', 'copilul', 'pisica', 'si', 'mananca', 'peste', 'iaurt']

In [118]:
from collections import Counter

In [121]:
matrice = []
for tok in tokens:
    counts = Counter(tok)
    print("counts", counts)
    matrice.append([counts[w] for w in words])
matrice

counts Counter({'pisica': 1, 'mananca': 1, 'peste': 1})
counts Counter({'copilul': 1, 'mananca': 1, 'iaurt': 1})
counts Counter({'copilul': 1, 'si': 1, 'pisica': 1, 'se': 1, 'joaca': 1})


[[0, 0, 0, 1, 0, 1, 1, 0], [0, 0, 1, 0, 0, 1, 0, 1], [1, 1, 1, 1, 1, 0, 0, 0]]

In [123]:
pd.DataFrame(matrice, columns=words)

,joaca,se,copilul,pisica,si,mananca,peste,iaurt
0,0,0,0,1,0,1,1,0
1,0,0,1,0,0,1,0,1
2,1,1,1,1,1,0,0,0


# 2.2 Varianta cu sklearn

In [124]:
from sklearn.feature_extraction.text   import CountVectorizer

In [126]:
bag_of_words_encoder = CountVectorizer()
bag_of_words_encoder

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [127]:
corpus = [
    "Pisica mananca peste",
    "Copilul mananca iaurt",
    "Copilul si pisica se joaca",
]

In [129]:
x_bow = bag_of_words_encoder.fit_transform(corpus)
x_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 11 stored elements and shape (3, 8)>

In [134]:
pd.DataFrame(x_bow.toarray(), columns=bag_of_words_encoder.get_feature_names_out())

,copilul,iaurt,joaca,mananca,peste,pisica,se,si
0,0,0,0,1,1,1,0,0
1,1,1,0,1,0,0,0,0
2,1,0,1,0,0,1,1,1


# 3. TF-IDF (Term Frequency - Inverse Document Frequency)

In [135]:
from sklearn.feature_extraction.text   import TfidfVectorizer

In [136]:
tfidf_encoder = TfidfVectorizer()
tfidf_encoder

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [137]:
corpus = [
    "Pisica mananca peste",
    "Copilul mananca iaurt",
    "Copilul si pisica se joaca",
]

In [139]:
x_tfidf = tfidf_encoder.fit_transform(corpus)
x_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11 stored elements and shape (3, 8)>

In [140]:
pd.DataFrame(x_tfidf.toarray(), columns=tfidf_encoder.get_feature_names_out())

,copilul,iaurt,joaca,mananca,peste,pisica,se,si
0,0.000000,0.000000,0.000000,0.517856,0.680919,0.517856,0.000000,0.000000
1,0.517856,0.680919,0.000000,0.517856,0.000000,0.000000,0.000000,0.000000
2,0.373022,0.000000,0.490479,0.000000,0.000000,0.373022,0.490479,0.490479
